# Lab | Transformers
## Route every message to the right team

Yesterday you built a semantic search for a digital bank's support messages. The bank now wants to go one step further: **route** each new message automatically to the team that handles its intent, before any person reads it.

The support lead has picked the ten intents that agents mix up most often: card delivery, lost and compromised cards, PINs, failed top-ups and exchange rates. If a model can tell these apart, the rest will be easy. Your job is to compare three ways of doing it, and to recommend one:

+ a **zero-shot** model, which needs no labelled examples at all;
+ a **fine-tuned DistilBERT**, trained on the bank's labelled messages;
+ yesterday's **semantic search**, used as a classifier: the intent of the most similar past message.

**The data** is the Banking77 dataset from yesterday's lab, in `data/banking_train.csv` and `data/banking_test.csv`, filtered to the ten intents.

**This lab practises** using and fine-tuning pretrained transformers. You will use, from today's **Transformers and Hugging Face** lesson: `AutoTokenizer`, the zero-shot `pipeline`, `Dataset.map`, `AutoModelForSequenceClassification`, `TrainingArguments` and `Trainer`, and the comparison with the methods of Days 1 and 2.

**By the end, you can:**

1. Apply a zero-shot pipeline and fine-tune DistilBERT to classify customer messages by intent.
2. Evaluate both against a semantic-search baseline and justify which one to use.

Each challenge ends with a **checkpoint** that tells you what you should see. If your numbers are very different, fix that before moving on. Fine-tuning takes one to four minutes on a laptop CPU; on a slow computer, open the notebook in [Google Colab](https://colab.research.google.com) with a GPU runtime.

## Setup

You need `transformers`, `torch`, `datasets`, `accelerate` and `sentence-transformers`. If the import fails, install them from a terminal with `pip install transformers torch datasets accelerate sentence-transformers`. The first run downloads three models (about 500 MB in total) unless you already have them from the lesson.

In [5]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import logging
import warnings
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")

import time
import numpy as np
import pandas as pd
import torch

from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
import datasets
datasets.disable_progress_bars()
from sentence_transformers import SentenceTransformer

from sklearn.metrics import accuracy_score, classification_report

pd.set_option("display.max_colwidth", 90)
_ = torch.manual_seed(0)

In [2]:
intents = ["card_arrival", "card_delivery_estimate", "lost_or_stolen_card", "compromised_card", "card_not_working",
           "pin_blocked", "change_pin", "top_up_failed", "top_up_reverted", "exchange_rate"]

train = pd.read_csv("../data/banking_train.csv")
test = pd.read_csv("../data/banking_test.csv")
train = train[train["category"].isin(intents)].reset_index(drop=True)
test = test[test["category"].isin(intents)].reset_index(drop=True)
print(train.shape, test.shape)
train.sample(5, random_state=0)

(1185, 2) (400, 2)


,text,category
833,What do I do if I detect fraudulent use on my car?,compromised_card
567,I lost my card. Can you help me?,lost_or_stolen_card
950,Can my PIN changed remotely without visiting a bank?,change_pin
922,I need to make my card PIN a different number,change_pin
362,how long does it take to get my card i am in a rush,card_delivery_estimate


**Checkpoint:** `(1185, 2) (400, 2)`: 1,185 past messages to learn from, and 400 new messages (40 per intent) to test on.

## Challenge 1: look at the messages as tokens

Before choosing a model, check what the model will actually read.

1. Load the tokenizer of `distilbert/distilbert-base-uncased` and print the tokens of three messages: `"My card hasn't arrived yet"`, `"Someone used my card in Bulgaria!"` and `"Why did my top-up get reverted?"`.
2. Count the tokens of every training message (`len(tokenizer(text)["input_ids"])`) and print the `describe()` of the counts.

In [14]:
checkpoint = "distilbert/distilbert-base-uncased"

# 1. Tokenizer and the tokens of three messages
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
print("Vocabulary size:", tokenizer.vocab_size)

print(tokenizer.tokenize("My card hasn't arrived yet"))
print(tokenizer.tokenize("Someone used my card in Bulgaria!"))
print(tokenizer.tokenize("Why did my top-up get reverted?"))
# 2. Number of tokens per training message
messages = [
    "My card hasn't arrived yet",
    "Someone used my card in Bulgaria!",
    "Why did my top-up get reverted?"
]

token_counts = [
    len(tokenizer(text)["input_ids"])
    for text in messages
]

df = pd.DataFrame({
    "message": messages,
    "token_count": token_counts
})

print(df)
print(df["token_count"].describe())


Vocabulary size: 30522
['my', 'card', 'hasn', "'", 't', 'arrived', 'yet']
['someone', 'used', 'my', 'card', 'in', 'bulgaria', '!']
['why', 'did', 'my', 'top', '-', 'up', 'get', 'reverted', '?']
                             message  token_count
0         My card hasn't arrived yet            9
1  Someone used my card in Bulgaria!            9
2    Why did my top-up get reverted?           11
count     3.000000
mean      9.666667
std       1.154701
min       9.000000
25%       9.000000
50%       9.000000
75%      10.000000
max      11.000000
Name: token_count, dtype: float64


**Checkpoint:** `hasn't` becomes `hasn`, `'` and `t`, and `top-up` becomes `top`, `-` and `up`. The messages are short: a median of 13 tokens and a mean of about 15, and the longest has 66, including `[CLS]` and `[SEP]`.

**Question:** you will fine-tune with `max_length=64`. How many training messages does that cut, and why is a short `max_length` worth having?

Using max_length=64 truncates all training messages containing more than 64 tokens, using sum(count > 64 for count in token_counts) the exact number of phrases to be truncated can be calculated. A short 'max_length' makes training faster and reduces memory consumption. However, it may remove useful information from longer messages, so it is importatn to check how many phrases are affected before.

after applying the code below, only one phrase from the train set was longer than 64 tokens 

In [15]:
token_counts = [
    len(tokenizer(text)["input_ids"])
    for text in train["text"]
]

cut_messages = sum(count > 64 for count in token_counts)

print("Messages longer than 64 tokens:", cut_messages)
print("Percentage truncated:", cut_messages / len(token_counts) * 100)

Messages longer than 64 tokens: 1
Percentage truncated: 0.08438818565400844


In [16]:
print(sum(count > 64 for count in token_counts))

1


In [17]:
tokenizer(
    text,
    truncation=True,
    max_length=64
)

{'input_ids': [101, 2026, 4003, 8440, 1005, 1056, 3369, 2664, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

## Challenge 2: zero-shot routing

The zero-shot model needs no training, only a description of each intent in plain English. The quality of those descriptions matters: the model compares each message with each description.

1. Complete the dictionary `descriptions` with a short description for the last three intents. Seven are given.
2. Create the zero-shot pipeline with the model `MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33`.
3. Run it on all the test messages, with `candidate_labels=list(descriptions.values())` and `batch_size=16`, and time it.
4. Turn the top label of each result back into an intent and compute the accuracy.

In [25]:
descriptions = {
    "card_arrival": "my new card has not arrived",
    "card_delivery_estimate": "how long card delivery takes",
    "lost_or_stolen_card": "my card was lost or stolen",
    "compromised_card": "someone else used my card",
    "card_not_working": "my card does not work",
    "pin_blocked": "my PIN is blocked",
    "change_pin": "how to change my PIN",
    "top_up_reverted": "Why did my top-up get reverted?",
    "top_up_failed": "Top-up is not working",
    "exchange_rate": "Will my money be of equal value when I travel abroad?"
    }

# 2. Zero-shot pipeline
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33")

candidate_labels = list(descriptions.values())

description_to_intent = {
    description: intent
    for intent, description in descriptions.items()
}

# 3. Predictions for the test messages, timed

test_messages = test["text"].tolist()

start = time.perf_counter()

results = zero_shot(
    test_messages,
    candidate_labels=candidate_labels,
    batch_size=16
)

elapsed_time = time.perf_counter() - start

print(f"Time taken: {elapsed_time:.2f} seconds")



# 4. Intents and accuracy
predicted_intents = [
    description_to_intent[result["labels"][0]]
    for result in results
]

print(predicted_intents[:10])

true_intents = test["intent"].tolist()

accuracy = accuracy_score(true_intents, predicted_intents)

print(f"Zero-shot accuracy: {accuracy:.2%}")

KeyboardInterrupt: 

In [26]:
print(zero_shot.model.config._name_or_path)
print(zero_shot.device)

MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33
cpu


In [27]:
print("Test messages:", len(test_messages))
print("Candidate labels:", len(candidate_labels))

Test messages: 400
Candidate labels: 10


In [28]:
start = time.perf_counter()

test_results = zero_shot(
    test_messages[:10],
    candidate_labels=list(descriptions.values()),
    batch_size=16
)

elapsed = time.perf_counter() - start

print(f"10 messages took: {elapsed:.2f} seconds")

10 messages took: 26.17 seconds


In [29]:
print("CPU cores:", os.cpu_count())
print("PyTorch threads:", torch.get_num_threads())
print("GPU available:", torch.cuda.is_available())

CPU cores: 12
PyTorch threads: 6
GPU available: False


In [30]:
print(zero_shot.model.config._name_or_path)
print(zero_shot.device)
print(len(test_messages))
print(torch.get_num_threads())

MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33
cpu
400
6


**Checkpoint:** the 400 messages take a few seconds to a minute, and the accuracy is about 0.76 with these descriptions (your own descriptions for the last three intents will move it a few points).

In [ ]:
# 5. Which intents does the zero-shot model confuse? Show the crosstab of true and predicted intents.

**Question:** which intents does the zero-shot model handle well, and which does it confuse? Why is that hard to fix with better descriptions?

*Your answer here.*

## Challenge 3: fine-tune DistilBERT

Now use the bank's 1,185 labelled messages. Follow the steps of section 6 of the lesson; the skeleton below has the structure, and you fill in the `TODO` parts.

1. Map every intent to a number (`label2id`) and back (`id2label`).
2. Write `to_dataset(df)`: build a `datasets.Dataset` with the columns `text` and `label`, and `map` the tokenizer on it with `truncation=True, max_length=64`.
3. Load `AutoModelForSequenceClassification` with 10 labels, `id2label` and `label2id`.
4. Train for 3 epochs with a batch size of 16 and a learning rate of `5e-5`, evaluating after every epoch.

In [ ]:
# 1. Intents as numbers
labels = sorted(intents)
label2id = None   # TODO
id2label = None   # TODO

# 2. Datasets
def to_dataset(df):
    # TODO
    pass

train_data, test_data = to_dataset(train), to_dataset(test)

# 3. Model with a new classification layer
classifier = None   # TODO

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(prediction.label_ids, predicted)}

# 4. Training
args = TrainingArguments(
    output_dir="intent-classifier",
    # TODO: epochs, batch size, learning rate, evaluation after every epoch
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    dataloader_pin_memory=False,
    seed=0,
)
trainer = None   # TODO

start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} seconds")

In [ ]:
# 5. Show the loss, the test loss and the test accuracy of every epoch (trainer.state.log_history)

**Checkpoint:** training takes one to four minutes on a CPU (a few seconds on a GPU). The test accuracy is about 0.92 after the first epoch and about 0.94 after the third, and both the training loss and the test loss fall at every epoch.

**Question:** the classification layer started with random weights. Why does the model still reach a good accuracy after a single pass over 1,185 messages?

*Your answer here.*

## Challenge 4: compare with yesterday's semantic search

Yesterday's semantic search can also classify: give a new message the intent of the most similar past message. Use it as the baseline.

1. Encode the training and test messages with `SentenceTransformer("all-MiniLM-L6-v2")` and `normalize_embeddings=True`.
2. For each test message, take the intent of the training message with the highest dot product, and compute the accuracy.
3. Get the predictions of the fine-tuned model on the test set (`trainer.predict(test_data)`, then `argmax` and `id2label`).
4. Build a DataFrame with one row per method (zero-shot, semantic search, fine-tuned DistilBERT), with the accuracy and the number of labelled messages each one used.

In [ ]:
# 1. Embeddings


# 2. Semantic search as a classifier


# 3. Fine-tuned predictions


# 4. Comparison table

**Checkpoint:** zero-shot about 0.76, semantic search about 0.92, fine-tuned DistilBERT about 0.94. Your fine-tuned accuracy can differ by a point or two from run to run and from computer to computer.

In [ ]:
# 5. Print the classification report of the fine-tuned model, and the 6 most frequent (true, predicted) mistakes

**Question:** where does the fine-tuned model still make mistakes? Are they model errors, or messages that a person would also find ambiguous?

*Your answer here.*

## Challenge 5: route new messages and recommend a method

1. Call `trainer.model.eval()`, then wrap the fine-tuned model in a `text-classification` pipeline (`model=trainer.model, tokenizer=tokenizer`).
2. Route five new messages: four that you write yourself, one for each of four different intents, plus `"I don't know what happened, the app just says error"`. Print each message with its predicted intent and score.

In [ ]:
# 1. Pipeline


# 2. Five new messages

**Checkpoint:** the messages that clearly belong to one intent get the right one with a high score (0.9 or more for the card that has not arrived and for the blocked PIN). Messages that are vague, or that sit between intents, get a lower score: in the reference solution, the unknown payment in a shop goes to `card_not_working` with a score of only 0.26, and the app error message, which belongs to none of the ten intents, is still forced into one of them.

**Question:** the bank asks you which method to put in production. Write a short recommendation (4 to 6 sentences): which method, why, and what should happen to messages like the last one.

*Your answer here.*

## Extra: how many labels do you need?

Labelling messages costs the bank time. Find out how the fine-tuned model and the semantic search behave with fewer labels.

1. Write a function that keeps `n` random training messages per intent (`train.groupby("category").sample(n, random_state=0)`).
2. For `n` in 10 and 40, fine-tune a fresh DistilBERT on that subset (same settings as Challenge 3, but 6 epochs for `n=10` because there are so few batches) and compute the test accuracy. Run the semantic search with the same subset as the past messages.
3. Show the results in a table, and add the numbers of Challenge 4 for the full training set.

In [ ]:
# Your code here

**Conclusions:** *Your answer here.*